# Municipal water and sanitation access needs

Executed IDA117V analysis. Stats SA municipal Census 2011/2022 indicators. The 90% rule is a project planning assumption. This notebook reproduces the complete experiment using the accompanying source modules. Run from the extracted project folder or 03_Code. Test scores are not used for model selection.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
root = Path.cwd()
if root.name == '03_Code': root = root.parent
if (root / 'IDA117V_Project').exists(): root = root / 'IDA117V_Project'
assert (root / '03_Code/run_analysis.py').exists()
sys.path.insert(0, str(root / '03_Code'))
import run_analysis
print('Project:', root.name)

Project: IDA117V_Project


## Audit and reproduce all models

This cell rebuilds the data, performs EDA, fits the dummy and four substantive models, searches only the validation-selected champion, runs three ablations, locks candidates and evaluates the test set. Neural models use the supplied NumPy implementation.

In [2]:
run_analysis.main()

Training Dummy
Training Logistic regression
Training Random forest
Training MLP
Training Residual network
Validation champion: Residual network Comparator: MLP
{
  "seed": 42,
  "train_n": 127,
  "validation_n": 43,
  "test_n": 43,
  "champion": "Residual network",
  "comparator": "MLP",
  "search_space": {
    "width": [
      8,
      16
    ],
    "lr": [
      0.001,
      0.003
    ],
    "l2": [
      0.01,
      0.1
    ],
    "dropout": [
      0.1,
      0.25
    ]
  },
  "best_cv_configuration": {
    "parameters": {
      "dropout": 0.1,
      "l2": 0.01,
      "lr": 0.001,
      "width": 16
    },
    "cv_mean": 0.8780685329643722,
    "cv_sd": 0.08204079770931463,
    "fold_scores": [
      0.9625730994152046,
      0.8303287380699894,
      0.9307359307359306,
      0.7886363636363636
    ]
  },
  "optimisation_accepted": true,
  "final_parameters": {
    "dropout": 0.1,
    "l2": 0.01,
    "lr": 0.001,
    "width": 16
  },
  "pre_optimisation_validation": {
    "accuracy

In [3]:
audit = json.loads((root / '05_Results/data_audit.json').read_text())
print(json.dumps(audit, indent=2))

{
  "records": 213,
  "duplicate_codes": 0,
  "class_counts": {
    "1": 149,
    "0": 64
  },
  "missing_values": {
    "code": 0,
    "municipality": 0,
    "province": 0,
    "category": 0,
    "water_2011": 0,
    "water_2022": 0,
    "sanitation_2011": 0,
    "sanitation_2022": 0,
    "refuse_2011": 0,
    "refuse_2022": 0,
    "electricity_2011": 0,
    "electricity_2022": 0,
    "population_2011": 0,
    "households_2011": 0,
    "household_size_2011": 0,
    "population_2022": 0,
    "households_2022": 0,
    "household_size_2022": 0,
    "formal_share_2022": 0,
    "traditional_share_2022": 0,
    "informal_share_2022": 0,
    "other_share_2022": 0,
    "urban_share_2022": 0,
    "rdp_share_2022": 0,
    "rdp_eligible_denominator_2022": 0,
    "log_households_2022": 0,
    "household_growth_pct": 0,
    "water_backlog_pct_2022": 0,
    "sanitation_backlog_pct_2022": 0,
    "higher_need": 0
  },
  "excluded_2011_dwelling_discrepancies": 213,
  "source_sha256": {
    "MAP_TABLES

## Validation selection and final test results

Macro-F1 is the selection measure. The positive class is higher access need. PR AUC in the output denotes average precision.

In [4]:
print(pd.read_csv(root / '05_Results/validation_comparison.csv').to_string(index=False))
print(pd.read_csv(root / '05_Results/test_metrics.csv').round(4).to_string(index=False))

              model  train_macro_f1  train_val_gap  validation_accuracy  validation_macro_f1  validation_precision  validation_recall  validation_weighted_f1  validation_roc_auc  validation_pr_auc  cv_mean    cv_sd
              Dummy        0.412037       0.001078             0.697674             0.410959              0.697674           1.000000                0.573431            0.500000           0.697674 0.412023 0.005476
Logistic regression        0.919467       0.124229             0.813953             0.795238              0.923077           0.800000                0.819712            0.941026           0.974781 0.878081 0.057314
      Random forest        0.912099       0.088638             0.837209             0.823460              0.960000           0.800000                0.842938            0.951282           0.979611 0.869842 0.090743
                MLP        0.937363       0.096230             0.860465             0.841133              0.928571           0.866667       

In [5]:
summary = json.loads((root / '05_Results/summary.json').read_text())
print('Champion:', summary['champion'])
print('Final parameters:', summary['final_parameters'])
print('Exact McNemar:', summary['mcnemar'])
print(pd.read_csv(root / '05_Results/ablations.csv').to_string(index=False))

Champion: Residual network
Final parameters: {'dropout': 0.1, 'l2': 0.01, 'lr': 0.001, 'width': 16}
Exact McNemar: {'both_correct': 33, 'champion_only_correct': 3, 'comparator_only_correct': 0, 'both_wrong': 7, 'discordant': 3, 'exact_p': 0.25, 'alpha': 0.05, 'comparator': 'MLP', 'statistic_continuity_corrected': 1.3333333333333333, 'independence_caveat': 'Municipal spatial dependence may violate independent-pair assumptions; inference is exploratory.'}
                          experiment  validation_macro_f1     delta                                                                                 removed_features
Remove historical service indicators             0.713333 -0.180755                                       water_2011, sanitation_2011, refuse_2011, electricity_2011
                    Remove geography             0.800926 -0.093163                                                                               province, category
    Remove housing and urban context           

## Inference validation

The same saved pipeline is used by the interface. The three valid records and one invalid record exercise shared input validation.

In [6]:
from predict import predict as predict_frame
valid = pd.read_csv(root / '02_Data/demo_valid_inputs.csv')
print(predict_frame(valid).to_string(index=False))
try:
    predict_frame(pd.read_csv(root / '02_Data/demo_invalid_input.csv'))
except ValueError as error:
    print('Expected invalid-input error:', error)

   predicted_class  higher_need_probability                                                        explanation
 Lower access need                 0.034880    Estimated 2022 category: access at least 90% for both services.
Higher access need                 0.965666 Estimated 2022 category: access below 90% for water or sanitation.
Higher access need                 0.954094 Estimated 2022 category: access below 90% for water or sanitation.
Expected invalid-input error: water_2011: percentages must be between 0 and 100.


## Conclusions

Final test accuracy is 83.7%, macro-F1 0.811 and higher-need recall 86.7%. Exact McNemar p=0.25 gives no evidence of a significant improvement over MLP. The sample is small and spatial dependence limits inference. The model describes historical 2022 access, with uncalibrated scores. Consult the PDF report, saved figures and original tables for the full interpretation. Run the interface with `python -m streamlit run 03_Code/app.py`.